# NB15a — D.1 USDCHF Deep-Dive (FX Industrialization, Phase D Step 1)

**Lock-Basis:** [ANN-016 FX as Reference Blueprint](../docs/decisions/ANN-016-fx-as-reference-blueprint-industrialization-first.md) — Phase D.1 (Architecture-First, vor allen technical D.4–D.8).

## Mission

USDCHF zeigt in NB14f-v2 invertierte Filter-Staffelung (Aggressive 0.97 → Balanced 0.63 → Conservative **0.17**) — alle 3 anderen Pairs (GBPUSD/AUDUSD/USDCAD) sind sauber monoton gestaffelt. Wir wollen verstehen **warum**.

**Kernfragen:**
1. Warum invertiert USDCHF?
2. Session- / Feature- / Regime- / strukturelles Pair-Problem?
3. Kann Universal-FX-Logik korrigiert werden, oder braucht CHF legitimen Override?
4. Würde ein Override überhaupt die ANN-016 Override-Discipline-Kriterien erfüllen?

## Strikter Lock — Diagnostik, KEINE Overrides

Per [ANN-016 Lock 5](../docs/decisions/ANN-016-fx-as-reference-blueprint-industrialization-first.md) ist dieses Notebook **rein diagnostisch**. Wir implementieren keine Overrides hier — nur Befunde dokumentieren, Confidence-Scores liefern, Daten-Grundlage für D.2 (ANN-017) schaffen.

## Pipeline

| Section | Inhalt |
|---|---|
| 0 | Setup + Drive + Imports |
| 1 | Load Features + Train Production-Modell (seed=7, Pool-Expansion) |
| 2 | Per-Session Performance Decomposition (4 Pairs × 6 Sessions) |
| 3 | **Marginal Contribution Analysis (4-Kombo Filter-Ablation pro Pair)** |
| 4 | **Conditioned SHAP** (Global / Winning / Losing / Per-Session) |
| 5 | **Vol-Regime percentile-basiert** (low <30% / mid / high >70%) |
| 6 | Liquidity-Proxy (Volume-Z / RVOL pro Session pro Pair) |
| 7 | **DXY-Regime + Korrelations-Check** |
| 8 | **Auto-Diagnose mit Confidence-Scores** (per Pair) |
| 9 | Result-Persistence (`results/nb15a/`) |
| 10 | Auto-Push to GitHub |
| 11 | Final Verdict — feeds D.2 (ANN-017) |

## Comparison-Pairs

Wir analysieren USDCHF tiefer und vergleichen mit zwei Referenz-Pairs:
- **USDCHF** — der Outlier (warum bricht es?)
- **USDCAD** — funktioniert sauber (Conservative PF 1.59), ähnliche USD-Quote-Struktur
- **GBPUSD** — funktioniert am besten (Conservative PF 1.83), kontrastive Referenz

## Section 0 — Setup + Drive Mount + Module Sync

In [ ]:
import os, sys, subprocess, gc, json, importlib
from pathlib import Path
from datetime import datetime, timezone

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = '/content/drive/MyDrive/pace-algo'
    os.chdir(PROJECT_ROOT)
    subprocess.run(['rm', '-rf', '/tmp/pace-algo'])
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/ecoNC/pace-algo.git',
                    '/tmp/pace-algo'], check=True)
    subprocess.run(f'mkdir -p {PROJECT_ROOT}/core/eval {PROJECT_ROOT}/core/analysis {PROJECT_ROOT}/core/router',
                   shell=True)
    subprocess.run(f'cp -rf /tmp/pace-algo/core/. {PROJECT_ROOT}/core/', shell=True)
    subprocess.run(f"find {PROJECT_ROOT}/core -type d -name __pycache__ -exec rm -rf {{}} +", shell=True)
    expected = [
        f'{PROJECT_ROOT}/core/analysis/diagnostic_decomposer.py',
        f'{PROJECT_ROOT}/core/analysis/probability_diagnostic.py',
    ]
    missing = [p for p in expected if not Path(p).exists()]
    if missing:
        print('SYNC FAILED:', missing)
        raise SystemExit('Re-run Section 0')
    print('Core synced.')
else:
    PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('core'):
        del sys.modules[mod]
importlib.invalidate_caches()

RANDOM_SEED = 7   # Production-Seed aus NB14f-v2
import numpy as np
np.random.seed(RANDOM_SEED)

RUN_DATE      = datetime.now(timezone.utc).strftime('%Y-%m-%d')
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H-%M-%SZ')
try:
    GIT_COMMIT = subprocess.check_output(['git', '-C', PROJECT_ROOT, 'rev-parse', '--short', 'HEAD'],
                                          text=True).strip()
except Exception:
    GIT_COMMIT = 'unknown'
EXPERIMENT_ID = f'nb15a_{RUN_TIMESTAMP}_{GIT_COMMIT}'
print(f'EXPERIMENT_ID: {EXPERIMENT_ID}')
print(f'Production seed: {RANDOM_SEED}')

In [ ]:
if IN_COLAB:
    subprocess.run(['pip', 'install', '-q', 'lightgbm>=4.3', 'shap>=0.45', 'pyarrow>=15.0'],
                   capture_output=True)

import pandas as pd
import lightgbm as lgb
import shap
import warnings
warnings.filterwarnings('ignore')

from core import config as cfg
from core.config import (
    FX_TRAIN_SYMBOLS, FX_HOLDOUT_SYMBOLS,
    TRAIN_END, VAL_END, HTF_CONTEXT_TIMEFRAMES, DATA_RAW,
)
from core.train.dataset import walk_forward_split, binary_label_for_long, NON_FEATURE_COLS
from core.train.lgbm_trainer import train_lgbm
from core.analysis.probability_diagnostic import extract_premium_cluster
from core.analysis.diagnostic_decomposer import (
    per_session_metrics, filter_marginal_contributions,
    conditioned_shap_extract, shap_delta,
    vol_regime_percentile, per_regime_metrics,
    macro_regime_annotator, rolling_correlation,
    diagnose_pair_inversion, override_discipline_precheck,
    SESSIONS_UTC,
)

TF = '5m'
R_VALUE = 1.5
FOCUS_PAIRS = ['USDCHF', 'USDCAD', 'GBPUSD']   # USDCHF + 2 Referenz-Pairs
ALL_HOLDOUT = FX_HOLDOUT_SYMBOLS   # ['GBPUSD', 'AUDUSD', 'USDCHF', 'USDCAD']

OUTPUT_DIR = Path(PROJECT_ROOT) / 'results' / 'nb15a'
for sub in ('metrics', 'shap', 'summaries', 'config_snapshots', 'diagnose'):
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

print(f'Focus pairs:    {FOCUS_PAIRS}')
print(f'All hold-out:   {ALL_HOLDOUT}')
print(f'Train symbols:  {FX_TRAIN_SYMBOLS}')
print(f'Output:         {OUTPUT_DIR}')

## Section 1 — Train Production-Modell (seed=7, Pool-Expansion) + Extract Cluster

In [ ]:
if IN_COLAB:
    DATA_EXT = Path('/content/processed_v2')
    DATA_PROCESSED_LOCAL = Path('/content/processed')
    EXT_DRIVE_BACKUP = Path(PROJECT_ROOT) / 'data_cache' / 'processed_v2'
    DRIVE_BACKUP_PROCESSED = Path(PROJECT_ROOT) / 'data_cache' / 'processed'
    DATA_RAW_PATH = Path(PROJECT_ROOT) / 'data_cache' / 'raw'
    DATA_EXT.mkdir(parents=True, exist_ok=True)
    DATA_PROCESSED_LOCAL.mkdir(parents=True, exist_ok=True)
    if len(list(DATA_PROCESSED_LOCAL.glob('labels_*.parquet'))) == 0 and DRIVE_BACKUP_PROCESSED.exists():
        subprocess.run(['rsync', '-ah', f'{DRIVE_BACKUP_PROCESSED}/', f'{DATA_PROCESSED_LOCAL}/'])
    if len(list(DATA_EXT.glob('*_extended.parquet'))) == 0 and EXT_DRIVE_BACKUP.exists():
        subprocess.run(['rsync', '-ah', f'{EXT_DRIVE_BACKUP}/', f'{DATA_EXT}/'])
else:
    DATA_EXT = cfg.DATA_PROCESSED.parent / 'processed_v2'
    DATA_PROCESSED_LOCAL = cfg.DATA_PROCESSED
    DATA_RAW_PATH = cfg.DATA_RAW

def load_ext(sym, tf=TF):
    p = DATA_EXT / f'{sym}_{tf}_extended.parquet'
    if not p.exists():
        return None
    df = pd.read_parquet(p)
    if 'hit_bar_offset' not in df.columns:
        lp = DATA_PROCESSED_LOCAL / f'labels_{sym}_{tf}_R{R_VALUE}.parquet'
        if lp.exists():
            labels = pd.read_parquet(lp)
            if 'hit_bar_offset' in labels.columns:
                df = df.join(labels[['hit_bar_offset']], how='left')
        if 'hit_bar_offset' not in df.columns:
            df['hit_bar_offset'] = 24
        df['hit_bar_offset'] = df['hit_bar_offset'].fillna(24).astype('int32')
    return df

# Verify all pairs available
missing = [s for s in (FX_TRAIN_SYMBOLS + ALL_HOLDOUT) if load_ext(s) is None]
if missing:
    raise SystemExit(f'Missing extended for: {missing}. Run NB14f Section 1 erst.')
print('Alle Pairs verfuegbar.')

In [ ]:
FEATURE_COLS = [
    'hour_sin', 'dist_to_swing_low_atr', 'htf_1h_rsi_14', 'realized_vol_20',
    'atr_percentile_100', 'atr_pct', 'dist_to_swing_high_atr', 'volume_z_score',
    'ema_20_slope_atr', 'hour_cos', 'momentum_composite', 'rvol_20',
    'adx_14', 'ema_20_dist_atr', 'htf_1h_atr_percentile_100',
    'htf_ltf_agree_bull', 'htf_ltf_agree_bear', 'htf_ltf_counter_trend',
    'htf_ltf_alignment_score', 'ltf_rsi_minus_htf_rsi',
    'both_rsi_oversold', 'both_rsi_overbought', 'vol_pct_diff_htf',
    'both_high_vol', 'both_low_vol', 'pullback_in_bull', 'pullback_in_bear',
]

probe = load_ext(FX_TRAIN_SYMBOLS[0])
FEATURE_COLS = [c for c in FEATURE_COLS if c in probe.columns]
del probe
gc.collect()

# Stack train pool
frames = []
for s in FX_TRAIN_SYMBOLS:
    d = load_ext(s)
    if d is None or d.empty:
        continue
    float_cols = d.select_dtypes(include=['float64']).columns
    if len(float_cols) > 0:
        d = d.astype({c: 'float32' for c in float_cols}, copy=False)
    frames.append(d)
train_pool = pd.concat(frames, axis=0).sort_index().dropna(subset=FEATURE_COLS + ['label'])
del frames
gc.collect()

train_df, val_df, test_df = walk_forward_split(train_pool, TRAIN_END, VAL_END)
del train_pool, test_df   # TEST nicht benoetigt — wir evaluieren auf Hold-Outs
gc.collect()
print(f'Train rows: {len(train_df):,}  VAL: {len(val_df):,}')

In [ ]:
X_train = train_df[FEATURE_COLS].values.astype(np.float32)
y_train = binary_label_for_long(train_df['label']).values
X_val = val_df[FEATURE_COLS].values.astype(np.float32)
y_val = binary_label_for_long(val_df['label']).values

# Trainiere mit Production-Seed=7 (deterministic=True ist im train_lgbm default)
params = {
    'objective': 'binary', 'metric': 'binary_logloss',
    'num_leaves': 7, 'max_depth': 3, 'min_data_in_leaf': 200,
    'learning_rate': 0.05, 'num_iterations': 30, 'lambda_l2': 0.5,
    'feature_fraction': 0.85, 'bagging_fraction': 0.85, 'bagging_freq': 5,
    'is_unbalance': True, 'verbose': -1, 'n_jobs': -1,
    'seed': RANDOM_SEED, 'deterministic': True,
}
td = lgb.Dataset(X_train, label=y_train)
vd = lgb.Dataset(X_val, label=y_val, reference=td)
model = lgb.train(params, td, num_boost_round=30, valid_sets=[vd],
                   callbacks=[lgb.log_evaluation(period=0)])

proba_val = model.predict(X_val)
del td, vd, X_train, y_train, X_val, y_val, train_df, val_df
gc.collect()

# Extract Premium-Cluster (ANN-013 cluster-based detection)
cluster_info = extract_premium_cluster(proba_val)
PRODUCTION_CLUSTER = cluster_info['premium_cluster_value']
print(f'\nProduction-Cluster (seed={RANDOM_SEED}): {PRODUCTION_CLUSTER:.4f}')
print(f'Cluster size: {cluster_info["premium_cluster_size"]} ({cluster_info["premium_cluster_pct"]:.2f}%)')

In [ ]:
# Pre-compute Inferenz auf alle Hold-Out-Pairs
pair_data = {}
for sym in ALL_HOLDOUT:
    h = load_ext(sym)
    if h is None or h.empty:
        continue
    h = h.dropna(subset=FEATURE_COLS + ['label'])
    _, _, h_test = walk_forward_split(h, TRAIN_END, VAL_END)
    if h_test.empty:
        continue
    X = h_test[FEATURE_COLS].values.astype(np.float32)
    proba = model.predict(X)
    pair_data[sym] = {
        'df':      h_test,
        'X':       X,
        'proba':   proba,
        'labels':  h_test['label'].values,
        'mask':    proba >= PRODUCTION_CLUSTER,
        'n_signals': int((proba >= PRODUCTION_CLUSTER).sum()),
    }
    print(f'  {sym}: {len(h_test):,} test bars, {pair_data[sym]["n_signals"]} Premium-Signale')
    del h
    gc.collect()

## Section 2 — Per-Session Performance Decomposition

In [ ]:
session_rows = []
for sym, pd_data in pair_data.items():
    sess_df = per_session_metrics(
        pd_data['df'].index, pd_data['mask'], pd_data['labels'], R=R_VALUE,
    )
    sess_df['symbol'] = sym
    session_rows.append(sess_df)
session_full = pd.concat(session_rows, ignore_index=True)

# Wide view: Pair × Session → PF
pivot_pf = session_full.pivot(index='symbol', columns='session', values='pf').round(3)
pivot_pf = pivot_pf[['asia', 'london', 'ny', 'ldn_ny_kz', 'eu_open', 'us_close']]
print('=== PF per Session per Pair ===')
display(pivot_pf)

pivot_n = session_full.pivot(index='symbol', columns='session', values='n').astype(int)
pivot_n = pivot_n[['asia', 'london', 'ny', 'ldn_ny_kz', 'eu_open', 'us_close']]
print('\n=== n_trades per Session per Pair ===')
display(pivot_n)

## Section 3 — Marginal Contribution Analysis

Echte 2×2-Filter-Ablation: Base / +HTF only / +Session only / +HTF + Session. Pro Pair separat.

**Wir wollen exakt sehen welcher Layer USDCHF invertiert.**

In [ ]:
# Define filter masks (ANN-012 Filter-Stack)
def htf_confirm_mask_for_pair(df, htf_align_col='htf_1h_ema_alignment'):
    if htf_align_col not in df.columns:
        return np.ones(len(df), dtype=bool)
    # HTF-Confirm: 1h trend matches LTF direction (long: HTF bullish; short: HTF bearish)
    # For Premium=long-bias model: HTF-Confirm = 1h alignment > 0
    return (df[htf_align_col].fillna(0).values > 0)

def ny_session_mask_for_index(idx):
    hours = np.asarray(idx.hour)
    return (hours >= 13) & (hours < 22)

marg_rows = []
for sym, pd_data in pair_data.items():
    df = pd_data['df']
    htf_mask = htf_confirm_mask_for_pair(df)
    ny_mask = ny_session_mask_for_index(df.index)
    marg_df = filter_marginal_contributions(
        proba=pd_data['proba'],
        cluster_cutoff=PRODUCTION_CLUSTER,
        timestamps=df.index,
        labels_triple=pd_data['labels'],
        htf_confirm_mask=htf_mask,
        session_mask=ny_mask,
        R=R_VALUE,
    )
    marg_df['symbol'] = sym
    marg_rows.append(marg_df)
marg_full = pd.concat(marg_rows, ignore_index=True)

# Wide view: Pair × Config → PF
marg_pivot_pf = marg_full.pivot(index='symbol', columns='config', values='pf').round(3)
marg_pivot_pf = marg_pivot_pf[['base', 'base_+htf', 'base_+session', 'base_+htf_+sess']]
marg_pivot_n = marg_full.pivot(index='symbol', columns='config', values='n').astype(int)
marg_pivot_n = marg_pivot_n[['base', 'base_+htf', 'base_+session', 'base_+htf_+sess']]

print('=== PF per Filter-Config per Pair ===')
display(marg_pivot_pf)
print('\n=== n_trades per Filter-Config per Pair ===')
display(marg_pivot_n)

# Delta-Sicht: welcher Filter zerstoert die Edge?
delta_pivot = marg_full.pivot(index='symbol', columns='config', values='pf_delta_vs_base').round(3)
delta_pivot = delta_pivot[['base_+htf', 'base_+session', 'base_+htf_+sess']]
print('\n=== PF-Delta vs Base per Pair (negative = Filter zerstoert Edge) ===')
display(delta_pivot)


## Section 4 — Conditioned SHAP

Nicht nur globale SHAPs sondern:
- **Global SHAP** auf alle Test-Bars
- **Winning trades SHAP** (label == +1)
- **Losing trades SHAP** (label == -1)
- **Session-conditioned SHAP** (NY / London / Asia)

Plus **Signed-Delta-Vergleich** Winning vs Losing — wo invertieren die Feature-Signaturen?

In [ ]:
explainer = shap.TreeExplainer(model)

# Sample auf Premium-Signale (sonst zu viel)
shap_records = {}
for sym in FOCUS_PAIRS:
    if sym not in pair_data:
        continue
    pd_data = pair_data[sym]
    mask = pd_data['mask']
    if mask.sum() == 0:
        continue
    # Sample max 3000 Premium-Signal-Bars
    idx_sig = np.where(mask)[0]
    if len(idx_sig) > 3000:
        idx_sig = np.random.choice(idx_sig, 3000, replace=False)
    X_sample = pd_data['X'][idx_sig]
    labels_sample = pd_data['labels'][idx_sig]
    hours = pd_data['df'].index[idx_sig].hour

    shap_vals = explainer.shap_values(X_sample)
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1] if len(shap_vals) > 1 else shap_vals[0]

    # Global
    g_mask = np.ones(len(X_sample), dtype=bool)
    g_shap = conditioned_shap_extract(shap_vals, FEATURE_COLS, g_mask, top_k=15)

    # Winning vs Losing
    w_mask = (labels_sample == 1)
    l_mask = (labels_sample == -1)
    w_shap = conditioned_shap_extract(shap_vals, FEATURE_COLS, w_mask, top_k=15)
    l_shap = conditioned_shap_extract(shap_vals, FEATURE_COLS, l_mask, top_k=15)

    # Session-conditioned
    ny_mask = (hours >= 13) & (hours < 22)
    ldn_mask = (hours >= 8) & (hours < 17)
    asia_mask = (hours >= 23) | (hours < 8)
    ny_shap = conditioned_shap_extract(shap_vals, FEATURE_COLS, ny_mask, top_k=15)
    ldn_shap = conditioned_shap_extract(shap_vals, FEATURE_COLS, ldn_mask, top_k=15)
    asia_shap = conditioned_shap_extract(shap_vals, FEATURE_COLS, asia_mask, top_k=15)

    # Deltas
    delta_win_loss = shap_delta(w_shap, l_shap, label_a='win', label_b='loss')
    delta_ny_ldn = shap_delta(ny_shap, ldn_shap, label_a='ny', label_b='ldn')

    shap_records[sym] = {
        'global':         g_shap,
        'winning':        w_shap,
        'losing':         l_shap,
        'ny':             ny_shap,
        'london':         ldn_shap,
        'asia':           asia_shap,
        'delta_win_loss': delta_win_loss,
        'delta_ny_ldn':   delta_ny_ldn,
    }
    print(f'\n=== {sym} — Top-5 Global SHAP ===')
    display(g_shap.head(5))
    print(f'=== {sym} — Top-5 |Signed Delta| Winning vs Losing ===')
    display(delta_win_loss[['feature', 'signed_delta', 'mean_signed_shap_win', 'mean_signed_shap_loss']].head(5))

## Section 5 — Vol-Regime Analyse (percentile-basiert)

Low < 30% / Mid / High > 70% — global classifiert auf `atr_percentile_100`. Pro Pair separat.

In [ ]:
regime_rows = []
for sym in FOCUS_PAIRS:
    if sym not in pair_data:
        continue
    pd_data = pair_data[sym]
    atr_pct = pd_data['df']['atr_percentile_100'].values if 'atr_percentile_100' in pd_data['df'].columns else None
    if atr_pct is None:
        print(f'  {sym}: atr_percentile_100 fehlt, skip')
        continue
    regimes = vol_regime_percentile(atr_pct, low_pctile=0.30, high_pctile=0.70, use_rolling=False)
    rg_df = per_regime_metrics(regimes, pd_data['mask'], pd_data['labels'], R=R_VALUE)
    rg_df['symbol'] = sym
    regime_rows.append(rg_df)
regime_full = pd.concat(regime_rows, ignore_index=True)

regime_pivot_pf = regime_full.pivot(index='symbol', columns='regime', values='pf').round(3)
regime_pivot_pf = regime_pivot_pf[[c for c in ['low', 'mid', 'high', 'unknown'] if c in regime_pivot_pf.columns]]
regime_pivot_n = regime_full.pivot(index='symbol', columns='regime', values='n').astype(int)
regime_pivot_n = regime_pivot_n[[c for c in ['low', 'mid', 'high', 'unknown'] if c in regime_pivot_n.columns]]

print('=== PF per Vol-Regime per Pair ===')
display(regime_pivot_pf)
print('\n=== n_trades per Vol-Regime per Pair ===')
display(regime_pivot_n)

## Section 6 — Liquidity-Proxy (Volume-Z / RVOL pro Session pro Pair)

In [ ]:
liq_rows = []
for sym in FOCUS_PAIRS:
    if sym not in pair_data:
        continue
    df = pair_data[sym]['df']
    hours = df.index.hour
    for session_name, window in SESSIONS_UTC.items():
        if session_name not in ('asia', 'london', 'ny'):
            continue
        lo, hi = window
        in_sess = (hours >= lo) & (hours < hi) if lo < hi else (hours >= lo) | (hours < hi)
        sub = df[in_sess]
        if len(sub) == 0:
            continue
        liq_rows.append({
            'symbol': sym,
            'session': session_name,
            'volume_z_mean': float(sub['volume_z_score'].mean()) if 'volume_z_score' in sub.columns else np.nan,
            'volume_z_abs_mean': float(sub['volume_z_score'].abs().mean()) if 'volume_z_score' in sub.columns else np.nan,
            'rvol_mean':    float(sub['rvol_20'].mean()) if 'rvol_20' in sub.columns else np.nan,
        })
liq_df = pd.DataFrame(liq_rows).round(3)
print('=== Liquidity Proxy per Session per Pair ===')
display(liq_df.pivot(index='symbol', columns='session', values='rvol_mean'))
print('\n=== |volume_z_score| per Session per Pair (Volatility proxy) ===')
display(liq_df.pivot(index='symbol', columns='session', values='volume_z_abs_mean'))

## Section 7 — DXY-Regime + Korrelations-Check

Hypothese: USDCHF verhält sich eher wie Macro-/Risk-Rotation als klassisches USD-Momentum.

**Test 1:** Per-DXY-Regime (bull/bear/sideways) Performance pro Pair  
**Test 2:** Rolling Correlation USDCHF mit den anderen Pairs (auf daily close-Basis)

In [ ]:
# Lade DXY aus Raw-Macro-File (yahoo macro)
macro_path = DATA_RAW_PATH / 'macro_daily.parquet'
if macro_path.exists():
    macro = pd.read_parquet(macro_path)
    if not macro.empty and 'DXY_close' in macro.columns:
        dxy = macro['DXY_close'].copy()
        if dxy.index.tz is None:
            dxy.index = pd.to_datetime(dxy.index).tz_localize('UTC')
        macro_available = True
        print(f'DXY series available: {dxy.index.min()} to {dxy.index.max()}, n={len(dxy)}')
    else:
        macro_available = False
        print('DXY_close fehlt im macro_daily.parquet')
else:
    macro_available = False
    dxy = pd.Series(dtype=float)
    print(f'macro_daily.parquet nicht vorhanden ({macro_path}) — Section 7 wird limited')

# Per-Pair DXY-Regime
macro_regime_rows = []
if macro_available:
    for sym in FOCUS_PAIRS:
        if sym not in pair_data:
            continue
        pd_data = pair_data[sym]
        regimes = macro_regime_annotator(pd_data['df'].index, dxy, rolling_days=20,
                                          bull_threshold=0.02, bear_threshold=-0.02)
        rg_df = per_regime_metrics(regimes, pd_data['mask'], pd_data['labels'], R=R_VALUE)
        rg_df = rg_df.rename(columns={'regime': 'dxy_regime', 'share_of_signals': 'share'})
        # rename to better column names
        rg_df['dxy_regime'] = rg_df['dxy_regime'].replace({'low': 'bear_or_unmapped',
                                                            'mid': 'sideways_or_unmapped',
                                                            'high': 'bull_or_unmapped'})
        # Actually for macro we have bull/bear/sideways labels — annotator returns those directly
        rg_df['symbol'] = sym
        macro_regime_rows.append(rg_df)
    macro_regime_full = pd.concat(macro_regime_rows, ignore_index=True) if macro_regime_rows else pd.DataFrame()
else:
    macro_regime_full = pd.DataFrame()

if not macro_regime_full.empty:
    macro_pivot = macro_regime_full.pivot(index='symbol', columns='dxy_regime', values='pf').round(3)
    print('=== PF per DXY-Regime per Pair ===')
    display(macro_pivot)
    macro_n = macro_regime_full.pivot(index='symbol', columns='dxy_regime', values='n').astype(int)
    print('\n=== n_trades per DXY-Regime per Pair ===')
    display(macro_n)

In [ ]:
# Rolling Correlation USDCHF mit anderen Pairs (auf daily close)
if 'USDCHF' in pair_data:
    chf_close = pair_data['USDCHF']['df']['close'] if 'close' in pair_data['USDCHF']['df'].columns else None
    if chf_close is not None:
        chf_daily = chf_close.resample('1D').last()
        corr_rows = []
        for sym in ('USDCAD', 'GBPUSD', 'AUDUSD', 'EURUSD'):
            if sym in pair_data:
                other_close = pair_data[sym]['df']['close'] if 'close' in pair_data[sym]['df'].columns else None
                if other_close is not None:
                    other_daily = other_close.resample('1D').last()
                    corr = rolling_correlation(chf_daily.pct_change(), other_daily.pct_change(), window=60)
                    corr_rows.append({
                        'pair_partner': sym,
                        'corr_mean': float(corr.mean()),
                        'corr_std':  float(corr.std()),
                        'corr_min':  float(corr.min()),
                        'corr_max':  float(corr.max()),
                    })
        if corr_rows:
            corr_df = pd.DataFrame(corr_rows).round(3)
            print('=== Rolling 60d Correlation USDCHF returns vs other pairs ===')
            display(corr_df)
        else:
            corr_df = pd.DataFrame()
    else:
        corr_df = pd.DataFrame()
        print('USDCHF close-Spalte fehlt')
else:
    corr_df = pd.DataFrame()
    print('USDCHF nicht in pair_data')

## Section 8 — Auto-Diagnose mit Confidence-Scores

Pro Pair: heuristische Hypothesen-Bewertung. **KEINE Statistik-Inferenz**, nur signal-strength-Score zur Priorisierung der manuellen Auswertung.

In [ ]:
diagnoses = {}
for sym in FOCUS_PAIRS:
    if sym not in pair_data:
        continue
    sess_sub = session_full[session_full['symbol'] == sym].drop(columns=['symbol'])
    marg_sub = marg_full[marg_full['symbol'] == sym].drop(columns=['symbol'])
    rg_sub = regime_full[regime_full['symbol'] == sym].drop(columns=['symbol']) if not regime_full.empty else None
    macro_sub = macro_regime_full[macro_regime_full['symbol'] == sym].drop(columns=['symbol']).rename(
        columns={'dxy_regime': 'regime'}
    ) if not macro_regime_full.empty else None
    shap_win_loss = shap_records.get(sym, {}).get('delta_win_loss')

    diag = diagnose_pair_inversion(
        per_session_df=sess_sub,
        marginal_df=marg_sub,
        per_regime_df=rg_sub,
        shap_winning_vs_losing=shap_win_loss,
        per_macro_regime_df=macro_sub,
    )
    diagnoses[sym] = diag
    print(f'\n=== {sym} ===')
    print(f'Top hypothesis: {diag["top_hypothesis"]}  (score {diag["top_score"]})')
    print('Hypothesis scores:')
    for k, v in sorted(diag['hypothesis_scores'].items(), key=lambda x: -x[1]):
        print(f'  {v:.3f}  {k}')
    print('Notes:')
    for n in diag['notes']:
        print(f'  - {n}')

In [ ]:
# Override-Discipline Precheck — kann der USDCHF-Bruch ueberhaupt einen legitimen Override
# rechtfertigen? Wir checken die 4 Kriterien aus ANN-016 Lock 5 (mit aktuellem Daten-Stand,
# wo wir bisher gekommen sind).

if 'USDCHF' in diagnoses:
    chf_diag = diagnoses['USDCHF']
    chf_marg = marg_full[marg_full['symbol'] == 'USDCHF']
    # Hypothetische Pre-Check: wir wissen noch nicht ob OOS-Lift erreichbar ist
    # — die Frage ist OB ein Override gerechtfertigt waere wenn wir D.2 weiter machen.
    base_pf = chf_marg[chf_marg['config'] == 'base']['pf'].values[0] if len(chf_marg[chf_marg['config'] == 'base']) else 0
    full_pf = chf_marg[chf_marg['config'] == 'base_+htf_+sess']['pf'].values[0] if len(chf_marg[chf_marg['config'] == 'base_+htf_+sess']) else 0
    potential_lift = base_pf - full_pf   # wenn wir Universal-Filter abschalten wuerden
    
    pre = override_discipline_precheck(
        statistical_proof=chf_diag['top_score'] > 0.5,   # confidence threshold
        market_structure_documented=False,                # NB15a sammelt nur Evidenz, ANN-017 dokumentiert
        oos_lift_pf=potential_lift,                       # hypothetisch — echtes OOS-Lift wird D.2 messen
        seeds_tested=1,                                    # bisher nur seed=7
        time_periods_tested=1,                            # nur ein Walk-Forward-Test
    )
    print('=== Override-Discipline Precheck (USDCHF) ===')
    print(f'all_passed: {pre["all_passed"]}')
    print('checks:')
    for k, v in pre['checks'].items():
        print(f'  {"\u2713" if v else "\u2717"} {k}')
    print(f'recommendation: {pre["recommendation"]}')
    print('\nNote: Diese Pre-Check ist EXPECTED FAIL — NB15a sammelt nur Evidenz.')
    print('ANN-017 (D.2) wird die fehlenden Kriterien adressieren (Multi-Seed, Multi-Period, OOS-Lift).')

## Section 9 — Result Persistence (`results/nb15a/`)

In [ ]:
# Persist CSVs
session_full.to_csv(OUTPUT_DIR / 'metrics' / f'per_session_{RUN_DATE}.csv', index=False)
marg_full.to_csv(OUTPUT_DIR / 'metrics' / f'marginal_contributions_{RUN_DATE}.csv', index=False)
if not regime_full.empty:
    regime_full.to_csv(OUTPUT_DIR / 'metrics' / f'per_vol_regime_{RUN_DATE}.csv', index=False)
if not liq_df.empty:
    liq_df.to_csv(OUTPUT_DIR / 'metrics' / f'liquidity_proxy_{RUN_DATE}.csv', index=False)
if not macro_regime_full.empty:
    macro_regime_full.to_csv(OUTPUT_DIR / 'metrics' / f'per_dxy_regime_{RUN_DATE}.csv', index=False)
if not corr_df.empty:
    corr_df.to_csv(OUTPUT_DIR / 'metrics' / f'rolling_correlation_usdchf_{RUN_DATE}.csv', index=False)

# SHAP-Records pro Pair
for sym, recs in shap_records.items():
    for label, df in recs.items():
        df.to_csv(OUTPUT_DIR / 'shap' / f'{sym}_shap_{label}_{RUN_DATE}.csv', index=False)

# Full snapshot
def _to_json_safe(d):
    if isinstance(d, dict):
        return {k: _to_json_safe(v) for k, v in d.items()}
    if isinstance(d, list):
        return [_to_json_safe(v) for v in d]
    if isinstance(d, (np.floating, np.integer)):
        return float(d)
    if isinstance(d, pd.DataFrame):
        return d.to_dict(orient='records')
    return d

snapshot = {
    'experiment_id':       EXPERIMENT_ID,
    'run_date':            RUN_DATE,
    'git_commit':          GIT_COMMIT,
    'production_seed':     RANDOM_SEED,
    'production_cluster':  float(PRODUCTION_CLUSTER),
    'cluster_info':        _to_json_safe(cluster_info),
    'focus_pairs':         FOCUS_PAIRS,
    'all_holdout':         list(ALL_HOLDOUT),
    'feature_cols':        FEATURE_COLS,
    'per_session':         _to_json_safe(session_full),
    'marginal_contributions': _to_json_safe(marg_full),
    'per_vol_regime':      _to_json_safe(regime_full),
    'liquidity_proxy':     _to_json_safe(liq_df),
    'per_dxy_regime':      _to_json_safe(macro_regime_full),
    'rolling_correlation_usdchf': _to_json_safe(corr_df),
    'shap_top_by_pair':    {sym: _to_json_safe(recs['global']) for sym, recs in shap_records.items()},
    'shap_winning_vs_losing_delta': {sym: _to_json_safe(recs['delta_win_loss']) for sym, recs in shap_records.items()},
    'diagnoses':           diagnoses,
    'override_precheck_usdchf': pre if 'pre' in dir() else None,
}
with open(OUTPUT_DIR / 'summaries' / f'nb15a_full_snapshot_{RUN_DATE}.json', 'w') as f:
    json.dump(snapshot, f, indent=2, default=str)

with open(OUTPUT_DIR / 'config_snapshots' / f'{EXPERIMENT_ID}_config.json', 'w') as f:
    json.dump({
        'experiment_id': EXPERIMENT_ID,
        'production_seed': RANDOM_SEED,
        'tf': TF,
        'focus_pairs': FOCUS_PAIRS,
    }, f, indent=2)

print(f'Results in {OUTPUT_DIR}')

## Section 10 — Auto-Push to GitHub

In [ ]:
import shutil
if not IN_COLAB:
    print('Local run — skip push.')
else:
    try:
        from google.colab import userdata
        GH_TOKEN = userdata.get('GITHUB_TOKEN') or userdata.get('GH_PAT')
    except Exception:
        GH_TOKEN = None
    if not GH_TOKEN:
        print('Kein GH-Token in Secrets — Results bleiben im Drive.')
    else:
        TMP_REPO = Path('/tmp/pace-algo-push')
        if TMP_REPO.exists():
            shutil.rmtree(TMP_REPO)
        subprocess.run(['git', 'clone', '--quiet',
                        f'https://{GH_TOKEN}@github.com/ecoNC/pace-algo.git', str(TMP_REPO)], check=True)
        subprocess.run(['git', '-C', str(TMP_REPO), 'config', 'user.name', 'ecoNC'], check=True)
        subprocess.run(['git', '-C', str(TMP_REPO), 'config', 'user.email',
                        'ecoNC@users.noreply.github.com'], check=True)
        copied = []
        for f in sorted(OUTPUT_DIR.rglob(f'*{RUN_DATE}*')):
            if not f.is_file():
                continue
            rel = f.relative_to(Path(PROJECT_ROOT) / 'results')
            dest = TMP_REPO / 'results' / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)
            copied.append(rel)
        for f in sorted((OUTPUT_DIR / 'config_snapshots').glob(f'*{EXPERIMENT_ID}*')):
            rel = f.relative_to(Path(PROJECT_ROOT) / 'results')
            dest = TMP_REPO / 'results' / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)
            copied.append(rel)
        subprocess.run(['git', '-C', str(TMP_REPO), 'pull', '--rebase', '--quiet', 'origin', 'main'], check=True)
        subprocess.run(['git', '-C', str(TMP_REPO), 'add', 'results/'], check=True)
        st = subprocess.run(['git', '-C', str(TMP_REPO), 'status', '--porcelain'], capture_output=True, text=True)
        if not st.stdout.strip():
            print('Nothing to commit.')
        else:
            top_hyp_chf = diagnoses.get('USDCHF', {}).get('top_hypothesis', '?')
            top_score_chf = diagnoses.get('USDCHF', {}).get('top_score', 0)
            msg = (f'NB15a D.1 USDCHF Deep-Dive {RUN_DATE} ({len(copied)} files)\n\n'
                   f'USDCHF top-hypothesis: {top_hyp_chf} ({top_score_chf:.2f})')
            subprocess.run(['git', '-C', str(TMP_REPO), 'commit', '-m', msg], check=True)
            sha = subprocess.run(['git', '-C', str(TMP_REPO), 'rev-parse', '--short', 'HEAD'],
                                  capture_output=True, text=True).stdout.strip()
            subprocess.run(['git', '-C', str(TMP_REPO), 'push', 'origin', 'main'], check=True)
            print(f'Pushed as ecoNC ({sha})')
        shutil.rmtree(TMP_REPO)

## Section 11 — Final Verdict-Frame

Nach dem Run analysiert Claude die JSON + CSVs und schreibt:

1. **`research/usdchf_deep_dive.md`** (NEU) — Strukturbefund-Doku mit allen Evidenzen
2. **ANN-017** (D.2 Universal-vs-Per-Pair Decision) — basierend auf top-hypothesis-Confidence
3. Decision-Tree:
   - Wenn top-hypothesis = `session_filter_inversion` mit Score > 0.7 → ANN-017 lockt **adaptive Session-Profile**
   - Wenn top-hypothesis = `macro_regime_dependency` > 0.6 → ANN-017 lockt **DXY-conditional Modell**
   - Wenn top-hypothesis = `structural_pair_issue` > 0.5 → ANN-017 lockt **Pair-Tiering als V1-Mechanik**
   - Wenn alle Scores < 0.5 → Pool-Expansion runde 2 (NZDUSD lessons learned anwenden) ODER mehr Features

**Wichtig:** D.2 wird datenbasiert entschieden, nicht Bauchgefühl. NB15a liefert die Eingangs-Daten. NB15a implementiert NIEMALS Overrides — das ist der Job von D.2 + späteren Phase-D-Schritten.